# 06 — Race-Condition-Safe Aggregation and Threshold Tuning

Companion notebook to `07-production-resilience-and-operational-engineering.md`. Two independent
resilience patterns from that chapter, both runnable and both demonstrated against a **failing**
naive baseline first, so the fix's effect is measured, not just asserted:

- **Part A** — the concurrency caveat and bug narrative 4: two evaluators (the synchronous Tier 1
  PII/toxicity gate and the asynchronous Tier 3 LLM-as-judge harmfulness classifier) racing on the
  same per-response flagged-count record. This section reproduces both failure shapes chapter 07
  names -- a **lost update** from a naive read-modify-write under genuine concurrency, and a
  **double count** when two different evaluators independently flag the same `response_id` -- and
  then implements both proposed fixes: an atomic increment, and (the chapter's preferred fix)
  counting `DISTINCT response_id` from a flags table with a unique constraint on
  `(response_id, evaluator_name)`.
- **Part B** — bug narrative 1: a percentage-based alert rule with no minimum-sample-count floor
  paging on-call for two borderline flags during a low-traffic overnight window. This section runs a
  Monte Carlo simulation across many simulated nights of varying traffic volume, measuring the actual
  false-positive alert rate of the naive rule versus the fixed rule (a minimum sample-count gate,
  falling back to a raw-count threshold below that floor) chapter 07 proposes.

Fully offline: numpy, pandas, and the standard library (including `threading`, used deliberately in
Part A to force a genuine race rather than only asserting one could happen) -- no real API keys, no
external calls.

In [1]:
import threading
import time
from dataclasses import dataclass, field
from typing import Dict, Set, Tuple

import numpy as np
import pandas as pd

rng = np.random.default_rng(11)
print("Imports OK")

Imports OK


## Part A.1 — Reproducing the lost update with a real race

`NaiveCounter.increment()` implements exactly the naive pattern chapter 07 calls out: `current =
get_flagged_count(...); set_flagged_count(..., current + 1)`. The `time.sleep()` between the read and
the write stands in for the round trip a real read-then-write against a metrics store would take --
long enough that Python's GIL genuinely hands control to another thread mid-operation, which is what
makes this a **real** race under real concurrency (many parallel threads), not just a sequential
demonstration that never exercises the interleaving chapter 07 says a proper concurrency test needs
to exercise.

In [2]:
class NaiveCounter:
    """The naive, race-prone application-level read-modify-write pattern from chapter 07."""
    def __init__(self):
        self.count = 0

    def increment(self):
        current = self.count          # READ
        time.sleep(0.0005)            # the gap a real DB round trip would take
        self.count = current + 1      # WRITE -- silently overwrites any concurrent writer's update


N_CONCURRENT_FLAGS = 60

naive_counter = NaiveCounter()
threads = [threading.Thread(target=naive_counter.increment) for _ in range(N_CONCURRENT_FLAGS)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"{N_CONCURRENT_FLAGS} evaluators each independently incremented the naive counter once.")
print(f"Expected count: {N_CONCURRENT_FLAGS}. Actual count: {naive_counter.count}.")
lost = N_CONCURRENT_FLAGS - naive_counter.count
print(f"Lost updates: {lost}")

assert naive_counter.count < N_CONCURRENT_FLAGS, (
    "expected the naive read-modify-write pattern to lose at least one update under real concurrency"
)
print()
print("Confirmed: the naive 'read current, write current + 1' pattern silently loses updates under "
      "genuine concurrent access -- this is bug narrative 4's root cause, reproduced with a real race, "
      "not just described.")

60 evaluators each independently incremented the naive counter once.
Expected count: 60. Actual count: 6.
Lost updates: 54

Confirmed: the naive 'read current, write current + 1' pattern silently loses updates under genuine concurrent access -- this is bug narrative 4's root cause, reproduced with a real race, not just described.


## Part A.2 — Fix #1: an atomic increment

Chapter 07's first proposed fix: replace the read-modify-write with an atomic SQL increment
(`UPDATE metrics_daily SET flagged_count = flagged_count + 1 WHERE ...`). A `threading.Lock` is the
in-process equivalent -- the increment becomes a single, indivisible operation no other thread can
observe halfway through. Re-running the identical concurrent workload against `AtomicCounter`
confirms it loses nothing.

In [3]:
class AtomicCounter:
    """Fix #1 (chapter 07): an atomic increment -- the in-process analog of
    'UPDATE ... SET flagged_count = flagged_count + 1'."""
    def __init__(self):
        self.count = 0
        self._lock = threading.Lock()

    def increment(self):
        with self._lock:
            current = self.count
            time.sleep(0.0005)   # same simulated round-trip delay as the naive version, on purpose
            self.count = current + 1


atomic_counter = AtomicCounter()
threads = [threading.Thread(target=atomic_counter.increment) for _ in range(N_CONCURRENT_FLAGS)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Expected count: {N_CONCURRENT_FLAGS}. Actual count: {atomic_counter.count}.")
assert atomic_counter.count == N_CONCURRENT_FLAGS, "the atomic increment must not lose any updates"
print("Confirmed: the atomic increment loses zero updates under the identical concurrent workload "
      "that lost updates above.")

Expected count: 60. Actual count: 60.
Confirmed: the atomic increment loses zero updates under the identical concurrent workload that lost updates above.


## Part A.3 — The other failure shape: a double count across two *different* evaluators

Chapter 07's bug narrative 4 also names the counterintuitive opposite failure: **both** increments
landing, because two different evaluators (Tier 1's PII gate and Tier 3's LLM-as-judge harmfulness
classifier) each independently decide the *same* `response_id` is flagged, and a naive "count of flag
events" model treats that as **two** flagged conversations instead of **one**. An atomic increment
alone doesn't fix this -- it correctly counts every flag *event*, but a flag event is not the same
thing as a flagged *conversation*.

In [4]:
def naive_flag_response(state: dict, response_id: str, evaluator_name: str):
    """Naive: every flag event increments a single shared counter, regardless of whether this
    response_id was already flagged by a different evaluator."""
    state["flagged_count"] = state.get("flagged_count", 0) + 1


naive_state = {}
naive_flag_response(naive_state, "resp-501", "tier1_pii_toxicity_gate")
naive_flag_response(naive_state, "resp-501", "tier3_llm_judge_harmfulness")   # SAME response, different evaluator

print(f"Naive flagged_count after two evaluators independently flag the SAME response: "
      f"{naive_state['flagged_count']}")
assert naive_state["flagged_count"] == 2, "reproducing the double-count: one conversation counted as two"
print("This is the double-count half of bug narrative 4 -- one genuinely flagged conversation is "
      "reported to alerting as two, inflating the flagged rate and feeding false alarms of the same "
      "shape as bug narrative 1 (Part B below).")

Naive flagged_count after two evaluators independently flag the SAME response: 2
This is the double-count half of bug narrative 4 -- one genuinely flagged conversation is reported to alerting as two, inflating the flagged rate and feeding false alarms of the same shape as bug narrative 1 (Part B below).


## Part A.4 — Fix #2 (chapter 07's preferred fix): `COUNT(DISTINCT response_id)` over a unique-constrained flags table

Instead of incrementing a counter at all, record each flag as a row in a `flags` table with a unique
constraint on `(response_id, evaluator_name)`, and report the count of **distinct flagged
`response_id`s**. This fixes *both* failure shapes at once: the unique constraint makes a retried
write from the same evaluator a no-op (no lost-update race possible, because there's nothing to
race -- an insert either adds a new row or is rejected as a duplicate), and counting distinct
`response_id`s means two different evaluators flagging the same response correctly collapses to one
flagged conversation, not two.

In [5]:
class FlagsTable:
    """Fix #2 (chapter 07, preferred): a flags table with a UNIQUE constraint on
    (response_id, evaluator_name), reporting COUNT(DISTINCT response_id) rather than
    incrementing a counter at all."""
    def __init__(self):
        self._rows: Set[Tuple[str, str]] = set()
        self._lock = threading.Lock()

    def flag(self, response_id: str, evaluator_name: str):
        with self._lock:
            self._rows.add((response_id, evaluator_name))   # set semantics == the unique constraint

    def distinct_flagged_response_count(self) -> int:
        return len({response_id for (response_id, _evaluator) in self._rows})


flags_table = FlagsTable()

# Two DIFFERENT evaluators flag the same response -- must count as ONE flagged conversation.
flags_table.flag("resp-501", "tier1_pii_toxicity_gate")
flags_table.flag("resp-501", "tier3_llm_judge_harmfulness")

# The SAME evaluator retries its write (e.g. a retried request after a transient timeout) -- must
# still be a no-op, not a second flag.
flags_table.flag("resp-501", "tier1_pii_toxicity_gate")

# A genuinely different response, flagged once.
flags_table.flag("resp-502", "tier1_pii_toxicity_gate")

print(f"Raw flag rows recorded: {len(flags_table._rows)}")
print(f"Distinct flagged response count: {flags_table.distinct_flagged_response_count()}")

assert len(flags_table._rows) == 3, "resp-501/tier1 (x2, deduped to 1) + resp-501/tier3 + resp-502/tier1 = 3 unique rows"
assert flags_table.distinct_flagged_response_count() == 2, "exactly two distinct flagged responses: resp-501 and resp-502"
print()
print("Confirmed: resp-501 (flagged twice, once by each evaluator, plus one retried duplicate write) "
      "correctly counts as ONE flagged conversation, and the retried write from tier1 didn't "
      "double-count either -- both failure shapes from bug narrative 4 are closed by the same fix.")

Raw flag rows recorded: 3
Distinct flagged response count: 2

Confirmed: resp-501 (flagged twice, once by each evaluator, plus one retried duplicate write) correctly counts as ONE flagged conversation, and the retried write from tier1 didn't double-count either -- both failure shapes from bug narrative 4 are closed by the same fix.


## Part A.5 — Confirming the fix holds under real concurrency, not just sequential calls

Re-running the earlier genuine-race workload (many threads, real GIL interleaving) against
`FlagsTable`, but this time simulating two evaluators racing on flagging the *same* small pool of
`response_id`s -- the exact scenario chapter 07 says a proper concurrency test needs to exercise
("fires both evaluators against the same `response_id` genuinely concurrently... and asserts the
final distinct-flagged-conversation count").

In [6]:
RESPONSE_POOL = [f"resp-{i}" for i in range(20)]     # 20 distinct responses in this window
EVALUATORS = ["tier1_pii_toxicity_gate", "tier3_llm_judge_harmfulness"]

concurrent_flags_table = FlagsTable()

def flag_with_delay(response_id, evaluator_name):
    time.sleep(0.0003)   # widen the race window the same way Part A.1 did
    concurrent_flags_table.flag(response_id, evaluator_name)

# Every response gets flagged by BOTH evaluators, fired as genuinely concurrent threads, plus each
# call is additionally duplicated (a retried write) to also exercise the same-evaluator race.
jobs = []
for rid in RESPONSE_POOL:
    for ev in EVALUATORS:
        jobs.append((rid, ev))
        jobs.append((rid, ev))   # duplicate/retried write

threads = [threading.Thread(target=flag_with_delay, args=job) for job in jobs]
for t in threads:
    t.start()
for t in threads:
    t.join()

result = concurrent_flags_table.distinct_flagged_response_count()
print(f"Responses in pool: {len(RESPONSE_POOL)}. Distinct flagged responses after concurrent, "
      f"duplicated flagging by both evaluators: {result}")

assert result == len(RESPONSE_POOL), (
    "every response in the pool was flagged by both evaluators (plus duplicate writes) -- the "
    "distinct-response_id count must equal the pool size exactly, under real concurrency"
)
print("Confirmed under genuine concurrent, duplicated flagging: no lost updates, no double counts.")

Responses in pool: 20. Distinct flagged responses after concurrent, duplicated flagging by both evaluators: 20
Confirmed under genuine concurrent, duplicated flagging: no lost updates, no double counts.


## Part B.1 — Reproducing bug narrative 1: a percentage rule with no sample-count floor

The exact scenario chapter 07 names: a flat "alert if flag rate > 20% in the trailing window" rule,
applied to a low-traffic overnight window. The true harmfulness-classifier flag rate is a low,
unremarkable **3%** throughout -- so *any* alert this simulation produces is, by construction, a false
positive. `simulate_window()` draws `n_responses` independent Bernoulli flags at the true rate, the
same way a small overnight sample would.

In [7]:
TRUE_FLAG_RATE = 0.03          # the assistant's real, steady-state harmfulness-flag rate
NAIVE_THRESHOLD = 0.20         # chapter 07's original "alert if > 20%" rule
MIN_SAMPLE_COUNT = 50          # chapter 07's proposed floor
RAW_COUNT_THRESHOLD = 5        # chapter 07's proposed fallback below the floor


def simulate_window(n_responses: int, true_rate: float = TRUE_FLAG_RATE) -> int:
    return int(rng.binomial(n=n_responses, p=true_rate))


def naive_alert(flagged: int, n: int, threshold: float = NAIVE_THRESHOLD) -> bool:
    if n == 0:
        return False
    return (flagged / n) > threshold


def gated_alert(flagged: int, n: int, threshold: float = NAIVE_THRESHOLD,
                 min_n: int = MIN_SAMPLE_COUNT, raw_count_threshold: int = RAW_COUNT_THRESHOLD) -> bool:
    if n >= min_n:
        return (flagged / n) > threshold
    return flagged >= raw_count_threshold


# The exact incident from chapter 07's bug narrative 1: 2 of 6 sampled responses flagged overnight.
incident_n, incident_flagged = 6, 2
print(f"Bug narrative 1 replayed: {incident_flagged}/{incident_n} = {incident_flagged/incident_n:.0%} flagged.")
print(f"Naive rule (>{NAIVE_THRESHOLD:.0%}, no sample floor):  alert = {naive_alert(incident_flagged, incident_n)}")
print(f"Gated rule (n>={MIN_SAMPLE_COUNT} else raw count>={RAW_COUNT_THRESHOLD}): alert = "
      f"{gated_alert(incident_flagged, incident_n)}")

assert naive_alert(incident_flagged, incident_n) is True, "the naive rule should page on-call here, as it did in the real incident"
assert gated_alert(incident_flagged, incident_n) is False, "the gated rule should NOT page on-call for 2 borderline flags"
print()
print("Confirmed: the exact bug-narrative-1 numbers page on-call under the naive rule and correctly "
      "stay silent under the gated rule.")

Bug narrative 1 replayed: 2/6 = 33% flagged.
Naive rule (>20%, no sample floor):  alert = True
Gated rule (n>=50 else raw count>=5): alert = False

Confirmed: the exact bug-narrative-1 numbers page on-call under the naive rule and correctly stay silent under the gated rule.


## Part B.2 — Monte Carlo: the actual false-positive rate across many nights of varying volume

A single replayed incident shows the fix works for *that* case. This section measures it properly:
simulate many independent "nights," each with a traffic volume drawn from a realistic mixture (a
low-traffic overnight-style window some of the time, a normal daytime-style window the rest of the
time), at the same steady 3% true flag rate throughout -- so every alert produced, under either rule,
is a genuine false positive, and the two rules' false-positive *rates* become directly comparable
numbers instead of a single anecdote.

In [8]:
N_NIGHTS = 5000
# Traffic volume mixture: ~30% of windows are low-traffic ("overnight"), ~70% are normal ("daytime").
is_overnight = rng.random(N_NIGHTS) < 0.30
overnight_n = rng.poisson(lam=6, size=N_NIGHTS).clip(min=1)
daytime_n = rng.poisson(lam=400, size=N_NIGHTS).clip(min=1)
n_responses = np.where(is_overnight, overnight_n, daytime_n)

flagged_counts = np.array([simulate_window(int(n)) for n in n_responses])

rows = []
for n, flagged, overnight in zip(n_responses, flagged_counts, is_overnight):
    rows.append({
        "n_responses": int(n),
        "flagged": int(flagged),
        "is_overnight": bool(overnight),
        "naive_alert": naive_alert(int(flagged), int(n)),
        "gated_alert": gated_alert(int(flagged), int(n)),
    })
nights_df = pd.DataFrame(rows)

naive_fp_rate = nights_df["naive_alert"].mean()
gated_fp_rate = nights_df["gated_alert"].mean()
naive_fp_overnight_rate = nights_df.loc[nights_df["is_overnight"], "naive_alert"].mean()
gated_fp_overnight_rate = nights_df.loc[nights_df["is_overnight"], "gated_alert"].mean()

print(f"Simulated {N_NIGHTS} nights at a steady true flag rate of {TRUE_FLAG_RATE:.0%} "
      f"(every alert below is a false positive by construction).\n")
print(f"{'Rule':<12}{'Overall FP rate':<20}{'Overnight-only FP rate':<25}")
print(f"{'naive':<12}{naive_fp_rate:<20.2%}{naive_fp_overnight_rate:<25.2%}")
print(f"{'gated':<12}{gated_fp_rate:<20.2%}{gated_fp_overnight_rate:<25.2%}")

assert gated_fp_rate < naive_fp_rate, "the gated rule must have a materially lower overall false-positive rate"
assert gated_fp_overnight_rate < naive_fp_overnight_rate, (
    "the gated rule's biggest improvement should be specifically on low-traffic overnight windows, "
    "matching chapter 07's bug narrative 1"
)
print()
print(f"The gated rule cuts the overnight false-positive rate from {naive_fp_overnight_rate:.1%} to "
      f"{gated_fp_overnight_rate:.1%} -- the concrete, measured version of chapter 07's fix, not just "
      "an assertion that a sample-count floor 'should help.'")

Simulated 5000 nights at a steady true flag rate of 3% (every alert below is a false positive by construction).

Rule        Overall FP rate     Overnight-only FP rate   
naive       1.00%               3.32%                    
gated       0.00%               0.00%                    

The gated rule cuts the overnight false-positive rate from 3.3% to 0.0% -- the concrete, measured version of chapter 07's fix, not just an assertion that a sample-count floor 'should help.'


## Tying it back

- **Part A** turns chapter 07's concurrency caveat into something that actually races (real threads,
  a real GIL-yielding delay between read and write) and shows both failure shapes bug narrative 4
  names -- a lost update, and a double count across two different evaluators -- getting closed by the
  same fix: stop counting events, start counting `DISTINCT response_id`s over a uniquely-constrained
  flags table.
- **Part B** turns bug narrative 1 into a measured false-positive rate rather than a single retold
  incident: the minimum-sample-count floor (falling back to a raw-count threshold below it) produces
  a materially lower false-positive rate specifically on the low-traffic overnight windows the
  original incident happened in -- without needing a larger floor to also suppress genuine signal
  during normal-volume daytime windows, since the gate only changes behavior below the floor.
- Both fixes share the same underlying lesson chapter 07 states as its takeaway: three of that
  chapter's four illustrative bugs live in the aggregation/alerting glue code, not in any individual
  evaluator's scoring logic -- which is exactly the layer both parts of this notebook test.